[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/tayalmanan28/MuJoCo-Tutorial/blob/main/tutorial/07_mjx.ipynb)

⚡ **本 notebook 需要 GPU 运行时。** 在 Colab 中：Runtime → Change runtime type → T4 GPU

# 教程 7：MJX——GPU 加速仿真

MJX 是 MuJoCo 的 JAX 后端，可在 GPU 上实现大规模并行仿真。
非常适合需要数千个环境的强化学习场景。

In [ ]:
!pip install -q mujoco mujoco-mjx jax[cuda12] mediapy

In [ ]:
import mujoco
from mujoco import mjx
import jax
import jax.numpy as jnp
import numpy as np
import mediapy as media
import time

print(f"JAX devices: {jax.devices()}")
print(f"MuJoCo version: {mujoco.__version__}")

## MJX 基础

MJX 会把 MuJoCo 模型转换为兼容 JAX 的 pytree，在 GPU 上运行：

1. 创建标准的 `MjModel`
2. 用 `mjx.put_model()` 转换为 MJX
3. 使用 `mjx.step()` 代替 `mujoco.mj_step()`

In [ ]:
xml = """
<mujoco>
  <option gravity="0 0 -9.81" timestep="0.002"/>
  <worldbody>
    <geom type="plane" size="5 5 0.1"/>
    <body pos="0 0 1">
      <joint type="free"/>
      <geom type="sphere" size="0.1" mass="1"/>
    </body>
  </worldbody>
</mujoco>
"""

# Standard MuJoCo model
mj_model = mujoco.MjModel.from_xml_string(xml)
mj_data = mujoco.MjData(mj_model)

# Convert to MJX
mjx_model = mjx.put_model(mj_model)
mjx_data = mjx.put_data(mj_model, mj_data)

print(f"MJX model type: {type(mjx_model)}")
print(f"MJX data qpos: {mjx_data.qpos}")

## JIT 编译的仿真步进

In [ ]:
# JIT compile the step function
jit_step = jax.jit(mjx.step)

# First call triggers compilation
mjx_data = jit_step(mjx_model, mjx_data)

# Benchmark: 1000 steps
t0 = time.time()
for _ in range(1000):
    mjx_data = jit_step(mjx_model, mjx_data)
mjx_data.qpos.block_until_ready()  # wait for GPU
elapsed = time.time() - t0

print(f"1000 steps in {elapsed*1000:.1f}ms ({1000/elapsed:.0f} steps/sec)")
print(f"Final height: {float(mjx_data.qpos[2]):.4f}")

## 批量（并行）仿真

MJX 的真正威力：用 `jax.vmap` 同时仿真数千个环境。

In [ ]:
batch_size = 4096

# Create batch of initial states with random velocities
rng = jax.random.PRNGKey(0)
random_vels = jax.random.normal(rng, (batch_size, 6)) * jnp.array([2, 2, 5, 0.5, 0.5, 0.5])

# Reset base data
mj_data_reset = mujoco.MjData(mj_model)
mjx_data_base = mjx.put_data(mj_model, mj_data_reset)

# Batch: each environment gets a different initial velocity
batched_data = jax.vmap(lambda v: mjx_data_base.replace(qvel=v))(random_vels)

print(f"Batched qpos shape: {batched_data.qpos.shape}")  # (4096, 7)
print(f"Simulating {batch_size} environments in parallel...")

In [ ]:
# Batched step function
@jax.jit
def batched_step(data):
    return jax.vmap(mjx.step, in_axes=(None, 0))(mjx_model, data)

# Simulate 500 steps for all environments
n_steps = 500

t0 = time.time()
data = batched_data
for _ in range(n_steps):
    data = batched_step(data)
data.qpos.block_until_ready()
elapsed = time.time() - t0

total_steps = batch_size * n_steps
print(f"{total_steps:,} total steps in {elapsed:.2f}s")
print(f"Throughput: {total_steps/elapsed:,.0f} steps/sec")
print(f"\nFinal heights — mean: {float(data.qpos[:, 2].mean()):.3f}, std: {float(data.qpos[:, 2].std()):.3f}")

## 渲染 MJX 的结果

MJX 在 GPU 上运行，但渲染用的是 CPU 上的 MuJoCo。把数据传回来：

In [ ]:
# Simulate one trajectory and render
mj_data_viz = mujoco.MjData(mj_model)
mjx_data_viz = mjx.put_data(mj_model, mj_data_viz)
mjx_data_viz = mjx_data_viz.replace(qvel=jnp.array([3.0, 0, 8.0, 0, 0, 0]))

renderer = mujoco.Renderer(mj_model, height=300, width=400)
frames = []

for i in range(300):
    mjx_data_viz = jit_step(mjx_model, mjx_data_viz)
    
    if i % 5 == 0:  # render every 5th step (for 60fps at dt=0.002)
        # Transfer back to CPU for rendering
        mjx.get_data_into(mj_data_viz, mj_model, mjx_data_viz)
        renderer.update_scene(mj_data_viz)
        frames.append(renderer.render().copy())

renderer.close()
media.show_video(frames, fps=30)

## 何时使用 MJX，何时使用 CPU MuJoCo

| 使用场景 | 建议 |
|----------|---------------|
| 单个环境、交互式 | CPU MuJoCo + `mujoco.viewer` |
| RL 训练（PPO、SAC） | MJX——批量仿真上千个环境 |
| 系统辨识 | MJX——可微物理 |
| 轨迹优化 | MJX——沿动力学做自动微分 |
| 接触丰富的复杂任务 | CPU MuJoCo（MJX 在接触方面有一些限制） |

### 主要区别
- MJX 支持大部分 MuJoCo 特性，但并非全部（参见[文档](https://mujoco.readthedocs.io/en/stable/mjx.html#feature-parity)）
- MJX 是**可微分的**——你可以沿物理仿真计算梯度
- 批量大小取 1024–8192 时，GPU 利用率最佳